# Explore CWRU raw signals and spectrograms

Quick sanity-check notebook: load a healthy and a faulty signal directly
from `data/raw/`, look at the raw waveform, then look at the log-STFT
spectrogram the pipeline actually feeds the autoencoder. Doesn't need a
trained model -- run this any time after `scripts/01_download_data.sh`.

If you're driving this notebook from a *different* Jupyter install (e.g.
the one in your other venv), register this project's venv as a kernel
first: `.venv/bin/python -m ipykernel install --user --name vibration-ad`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np

from vibration_ad.config import load_config
from vibration_ad.data.dataset import load_signal
from vibration_ad.features.spectrogram import windows_to_log_spectrograms
from vibration_ad.data.windows import make_windows

config = load_config(PROJECT_ROOT / "configs" / "default.yaml")
raw_dir = config.resolve(config.paths.raw_dir)
raw_dir

In [ ]:
healthy = load_signal(raw_dir / "1797_Normal.npz", config.data.sensor)
faulty = load_signal(raw_dir / "1797_IR_21_DE12.npz", config.data.sensor)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True, sharey=True)
n = 4000  # ~0.33s @ 12kHz
axes[0].plot(healthy[:n]); axes[0].set_title("Healthy (1797_Normal)")
axes[1].plot(faulty[:n]); axes[1].set_title("Inner-race fault, 21 mil (1797_IR_21_DE12)")
axes[1].set_xlabel("sample")
fig.tight_layout()

The faulty trace should show periodic sharp impulses (impact of the
damaged inner race hitting the rolling elements once per revolution-ish),
riding on top of a similar background level to the healthy signal -- often
hard to eyeball reliably in the raw waveform, which is exactly why we hand
the *frequency* representation to the model instead of the raw waveform.

In [ ]:
def one_spectrogram(signal):
    windows = make_windows(signal, config.windowing.segment_length, config.windowing.eval_stride)
    specs = windows_to_log_spectrograms(windows, config.data.fs_hz, config.stft)
    return specs[len(specs) // 2]  # a window from the middle of the file

spec_healthy = one_spectrogram(healthy)
spec_faulty = one_spectrogram(faulty)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, spec, title in zip(axes, [spec_healthy, spec_faulty], ["Healthy", "IR fault, 21 mil"]):
    im = ax.imshow(spec, aspect="auto", origin="lower", cmap="magma")
    ax.set_title(title)
    ax.set_xlabel("time frame")
    ax.set_ylabel("frequency bin")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()

Look for extra horizontal bands (energy concentrated at specific
frequencies/harmonics) in the faulty spectrogram that aren't present in
the healthy one -- that's the spectral signature the autoencoder is
trained to *not* be able to reconstruct.

Next: `scripts/02_train.sh`, `scripts/03_evaluate.sh`, then come back and
look at `outputs/figures/reconstruction_examples.png` to see the model's
own input/reconstruction/difference view of windows like these.